# 05 · Text data: tweets and other short texts

Use this notebook when each example is a short text (a tweet, a
review, a headline) with a label. It does for text what notebooks 01
and 02 do for images:

| Step | What happens |
|---|---|
| 1–4 | Setup, settings, read the CSV, look at the texts |
| 5 | Clean the texts: links and @names become tags |
| 6 | Remove duplicates (retweets!) and empty texts |
| 7 | Split into train / validation / test, and save the split |
| 8 | Build the vocabulary from the **training** texts only |
| 9 | Turn every text into a row of word numbers |
| 10–12 | Baseline, train, learning curves, experiment log |
| 13 | Test set, once, with the most confidently wrong texts |
| 14 | Save the files the Streamlit app needs |

**Two models** (set `MODEL`):

| `MODEL` | What it is |
|---|---|
| `"bag_of_words"` | Averages the word vectors and ignores word order. Your **baseline** |
| `"text_cnn"` | A 1D CNN: filters slide over 3 neighboring words and detect short phrases (the Week 11 ideas on a sequence of words) |

**Before you start:** put your CSV file in the `csc451_project` folder
of your Google Drive. It needs one column with the text and one with
the label.

## Step 1 · Setup

In [ ]:
# Run this cell first, every time you open the notebook.
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# CHANGE: the address of YOUR team's copy of this repo on GitHub
REPO_URL = "https://github.com/dsc-scsu/csc451-project-starter.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content") / Path(REPO_URL).stem
    if not REPO_DIR.exists():
        !git clone -q {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    !git pull -q
elif Path.cwd().name == "notebooks":
    os.chdir("..")  # on your own computer: work from the repo folder

sys.path.insert(0, "src")
print("Working in", Path.cwd())

## Step 2 · Settings (CHANGE)

| Setting | What to put |
|---|---|
| `CSV_NAME` | Your CSV file, in `PROJECT_FOLDER` on Drive |
| `TEXT_COLUMN`, `LABEL_COLUMN` | The names of the text and label columns |
| `MODEL` | `"bag_of_words"` or `"text_cnn"` |
| `MAX_WORDS` | The most words the model can know. Rarer words become `<unk>` (unknown) |
| `MIN_COUNT` | A word must appear this many times in the training texts to be known |
| `MAX_LEN` | Tokens kept per text. A 280-character tweet has at most about 60–70 tokens |
| `EMBED_DIM` | Numbers per word vector. If you change it, change it in `app/app_text.py` too |

In [ ]:
# CHANGE: settings for your project
PROJECT_FOLDER = "csc451_project"
CSV_NAME = "tweets.csv"
TEXT_COLUMN = "text"
LABEL_COLUMN = "label"

MODEL = "text_cnn"        # "bag_of_words" (baseline) or "text_cnn"
MAX_WORDS = 10000
MIN_COUNT = 2
MAX_LEN = 64
EMBED_DIM = 64

VAL_SIZE = 0.15
TEST_SIZE = 0.15
LR = 1e-3
EPOCHS = 30
PATIENCE = 5
BATCH_SIZE = 64
SEED = 0
OVERWRITE_SPLITS = False

RUN_NOTE = "first run: default settings"   # CHANGE every run
EVALUATE_TEST = False                      # True only at the very end

In [ ]:
# Where things are. You should not need to change this cell.
if IN_COLAB:
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
    DATA_DIR = Path("/content/data")      # fast disk, erased after
else:                                     # the session ends
    PROJECT = Path("project_files")       # on your own computer
    DATA_DIR = PROJECT / "unzipped"
OUT = PROJECT / "outputs"                 # results to keep, on Drive
OUT.mkdir(parents=True, exist_ok=True)
print("Project folder:", PROJECT)

import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

import text_utils as xu
import train_utils as tu

device = "cuda" if torch.cuda.is_available() else "cpu"

## Step 3 · Read the CSV
Texts with emoji or accents sometimes come in a different encoding. If reading fails, this cell tries a second one.

In [ ]:
try:
    df = pd.read_csv(PROJECT / CSV_NAME)
except UnicodeDecodeError:
    df = pd.read_csv(PROJECT / CSV_NAME, encoding="latin-1")
print(df.shape[0], "rows; columns:", list(df.columns))
df = df[[TEXT_COLUMN, LABEL_COLUMN]].rename(
    columns={TEXT_COLUMN: "text", LABEL_COLUMN: "label"})
df.head()

## Step 4 · Look at the texts
Read some texts from each class. Ask yourself: could a person tell the label from the text alone? Is anything in the text a giveaway that has nothing to do with the meaning (the same hashtag or link in every text of one class)?

In [ ]:
df = df.dropna()
df["label"] = df["label"].astype(str)
print(df["label"].value_counts().to_string())
print()
for label, rows in df.groupby("label"):
    print(f"--- {label}")
    for t in rows["text"].head(3):
        print("  ", t)

lengths = df["text"].astype(str).str.len()
print()
print(f"Characters per text: median {lengths.median():.0f}, "
      f"longest {lengths.max()}")

## Step 5 · Clean the texts

`xu.clean_text` lower-cases each text and replaces the parts that are
different in every tweet, so the model can learn from them:

- a link (`https://t.co/x7Qa`) becomes `<url>`
- a user name (`@nasa`) becomes `<user>`
- `RT` at the start (a retweet marker) is removed

Hashtags, emoji and punctuation are kept: they often carry meaning.

In [ ]:
df["clean"] = df["text"].map(xu.clean_text)
for before, after in zip(df["text"].head(5), df["clean"].head(5)):
    print("before:", before)
    print("after: ", after)
    print()

## Step 6 · Remove duplicates and empty texts

Tweets are copied often: retweets, bots, and the same text from many
accounts. If one copy lands in training and another in the test set,
the model is tested on a text it has already seen, and the test score
is too high (data leakage). Texts that appear with two different
labels are removed too: they cannot both be right.

In [ ]:
n_before = len(df)
df = df[df["clean"].str.len() > 0]
labels_per_text = df.groupby("clean")["label"].nunique()
conflicting = labels_per_text[labels_per_text > 1].index
df = df[~df["clean"].isin(conflicting)]
df = df.drop_duplicates(subset="clean").reset_index(drop=True)
print(n_before - len(df), "rows removed;", len(df), "texts left")
print(len(conflicting), "texts had two different labels")

counts = df["label"].value_counts()
print()
print(counts.to_string())
print(f"Largest class: {counts.max() / counts.sum():.1%} of the texts")

## Step 7 · Split, and save the split
A stratified split keeps the same share of each class in train, validation and test. It is saved to `outputs/text_splits.csv`, so every run uses the same texts.

In [ ]:
splits_file = OUT / "text_splits.csv"
if splits_file.exists() and not OVERWRITE_SPLITS:
    df = pd.read_csv(splits_file, keep_default_na=False,
                     dtype={"text": str, "label": str, "clean": str})
    print("Using the split saved earlier")
else:
    hold = VAL_SIZE + TEST_SIZE
    train_idx, rest_idx = train_test_split(
        df.index, test_size=hold, stratify=df["label"], random_state=SEED)
    rest = df.loc[rest_idx]
    val_idx, test_idx = train_test_split(
        rest.index, test_size=TEST_SIZE / hold, stratify=rest["label"],
        random_state=SEED)
    df.loc[train_idx, "split"] = "train"
    df.loc[val_idx, "split"] = "val"
    df.loc[test_idx, "split"] = "test"
    df.to_csv(splits_file, index=False)
    print("Saved", splits_file)

classes = sorted(df["label"].unique())
train_df = df[df["split"] == "train"]
val_df = df[df["split"] == "val"]
test_df = df[df["split"] == "test"]
print(pd.crosstab(df["label"], df["split"], margins=True))

## Step 8 · Build the vocabulary from the training texts

The **vocabulary** is the list of tokens the model knows; a token's
position in the list is its number. It is built from the training
texts only. Words that never appear in training become `<unk>`
(unknown) in validation and test, just as new words will in the app.

The share of unknown tokens in the validation texts shows how well
the vocabulary covers new text. A high share (over about 20%) means
too little training text, or `MIN_COUNT` too high.

In [ ]:
vocab = xu.build_vocab(train_df["clean"], MAX_WORDS, MIN_COUNT)
word_to_idx = {w: i for i, w in enumerate(vocab)}
print(len(vocab), "tokens in the vocabulary")
print("Most common:", vocab[2:22])

val_tokens = [t for text in val_df["clean"] for t in xu.tokenize(text)]
unknown = sum(t not in word_to_idx for t in val_tokens)
print(f"Unknown tokens in validation texts: {unknown / len(val_tokens):.1%}")

n_tokens = df["clean"].map(lambda t: len(xu.tokenize(t)))
print(f"Tokens per text: median {n_tokens.median():.0f}, "
      f"95% have at most {n_tokens.quantile(0.95):.0f} "
      f"(MAX_LEN = {MAX_LEN})")

## Step 9 · Turn every text into a row of word numbers
Each text becomes exactly `MAX_LEN` numbers: longer texts are cut, shorter ones are padded with 0 (`<pad>`). Then the rows go into DataLoaders, as in Week 7.

In [ ]:
def make_loader(part, shuffle):
    x = xu.encode_all(part["clean"], word_to_idx, MAX_LEN)
    y = torch.tensor([classes.index(c) for c in part["label"]])
    return DataLoader(TensorDataset(x, y), batch_size=BATCH_SIZE,
                      shuffle=shuffle)

train_dl = make_loader(train_df, shuffle=True)
val_dl = make_loader(val_df, shuffle=False)
test_dl = make_loader(test_df, shuffle=False)

example = train_df["clean"].iloc[0]
print("text:   ", example)
print("tokens: ", xu.tokenize(example)[:12])
print("numbers:", xu.encode(example, word_to_idx, MAX_LEN)[:12], "...")

## Step 10 · Baseline
Always answering the largest training class. Your models must beat it. Then run once with `MODEL = "bag_of_words"` and once with `"text_cnn"`, and compare them in the log.

In [ ]:
largest = train_df["label"].value_counts().idxmax()
baseline_val = (val_df["label"] == largest).mean()
print(f"Always answering '{largest}': {baseline_val:.1%} "
      "validation accuracy")

## Step 11 · Train
The best epoch by validation accuracy is saved to `outputs/best_text.pt`.

In [ ]:
tu.set_seed(SEED)
model = xu.make_text_model(MODEL, len(vocab), len(classes),
                           EMBED_DIM).to(device)
print(f"{MODEL}: {tu.count_parameters(model):,} parameters")
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=LR)
checkpoint = OUT / "best_text.pt"
history = tu.fit(model, train_dl, val_dl, loss_fn, opt, device,
                 EPOCHS, checkpoint, PATIENCE)

## Step 12 · Learning curves and experiment log

In [ ]:
hist = pd.DataFrame(history)
best_val = hist["val_acc"].max()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist["epoch"], hist["train_loss"], label="train")
ax1.plot(hist["epoch"], hist["val_loss"], label="validation")
ax1.set_xlabel("epoch")
ax1.set_ylabel("loss")
ax1.legend()
ax2.plot(hist["epoch"], hist["train_acc"], label="train")
ax2.plot(hist["epoch"], hist["val_acc"], label="validation")
ax2.axhline(baseline_val, color="gray", linestyle="--", label="baseline")
ax2.set_xlabel("epoch")
ax2.set_ylabel("accuracy")
ax2.legend()
plt.tight_layout()
plt.savefig(OUT / f"learning_curves_{MODEL}.png", dpi=150)
plt.show()

tu.log_run(OUT / "experiments_text.csv", note=RUN_NOTE, model=MODEL,
           max_words=MAX_WORDS, min_count=MIN_COUNT, max_len=MAX_LEN,
           embed_dim=EMBED_DIM, lr=LR, epochs_run=len(hist), seed=SEED,
           baseline_val=round(baseline_val, 4),
           best_val_acc=round(best_val, 4))
pd.read_csv(OUT / "experiments_text.csv")

## Step 13 · Test set (once, at the end)
Set `EVALUATE_TEST = True` only for your final model. The table of the most confidently wrong texts is the start of your error analysis: are they mislabeled, sarcastic, too short, or full of words the model has never seen?

In [ ]:
if not EVALUATE_TEST:
    print("Skipped. Set EVALUATE_TEST = True for your final model.")
else:
    y_true, y_pred, probs = tu.predict_all(model, test_dl, device)
    baseline_test = (test_df["label"] == largest).mean()
    print(f"Test accuracy: {(y_true == y_pred).mean():.1%}   "
          f"(baseline {baseline_test:.1%})")
    cm = confusion_matrix(y_true, y_pred, labels=range(len(classes)))
    ConfusionMatrixDisplay(cm, display_labels=classes).plot(
        xticks_rotation=45, colorbar=False)
    plt.title("Test set: rows are the true class")
    plt.tight_layout()
    plt.savefig(OUT / "text_confusion_matrix.png", dpi=150)
    plt.show()

    wrong = pd.DataFrame({
        "text": test_df["text"].to_numpy(),
        "true": [classes[i] for i in y_true],
        "said": [classes[i] for i in y_pred],
        "confidence": probs.max(axis=1).round(2),
    })
    wrong = wrong[wrong["true"] != wrong["said"]]
    pd.set_option("display.max_colwidth", 200)
    display(wrong.sort_values("confidence", ascending=False).head(10))

## Step 14 · Files for the Streamlit app

`app/app_text.py` needs `best_text.pt` and `text_info.json`. The
info file holds the model name, the class names, the vocabulary and
`MAX_LEN`, so the app turns a new text into numbers exactly as
training did. Copy both into the repo's `app/` folder.

In [ ]:
info = {"model": MODEL, "classes": classes, "vocab": vocab,
        "max_len": MAX_LEN, "embed_dim": EMBED_DIM}
with open(OUT / "text_info.json", "w") as f:
    json.dump(info, f)

if IN_COLAB:
    from google.colab import files
    files.download(str(checkpoint))
    files.download(str(OUT / "text_info.json"))
else:
    print("Copy into app/:", checkpoint, OUT / "text_info.json")